# Principal-component model in a 3-stock market

In this script we show how to use principal-component LFM to perform dimension reduction of risk
in a 3-dimensional stock market

In [1]:
from numpy import array, average, cov, eye, r_, sqrt, sum
from numpy.linalg import eigh, matrix_rank
from pandas import read_csv, to_datetime

# r_ is numpy's shorthand for stacking arrays row-wise.
# It's used at the end to build the residual-plus-factor matrix

In [2]:
# find the ROOT & DATA folder.

from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
DATA_DIR = ROOT / "data"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))  # lets 'from arpym...' work from any subfolder

Load Data

From database db_stocks_sp we upload the stocks values

In [13]:
# header=1 means the column names are on the files 2nd row
# usecols picks 3 stocks, pandas keeps them in the files column order
db_stocks_sp = read_csv(DATA_DIR/"db_stocks_sp.csv", parse_dates=True, date_format='mixed',
                        index_col=0, header=1, usecols=['date', 'CVC', 'AON', 'CSCO'])

# stock values
# date filter taking data from 16th-30th March 2012
# .T transposes again so v_stocks is stocks vs days (3x11)
v_stocks = array(db_stocks_sp[(db_stocks_sp.index >= to_datetime('16-03-2012', dayfirst=True))&
                              (db_stocks_sp.index <= to_datetime('30-03-2012', dayfirst=True))].values).T

# confirms which row is which stock visually
print(db_stocks_sp.columns)

Index(['AON', 'CSCO', 'CVC'], dtype='str')


1. Principal-component model in a 3-stock market

Targets, mean, covariance

In [5]:
# target variables
# [:, 1:] -> todays
# [:, :-1] -> yesterdays
x = v_stocks[:, 1:]/v_stocks[:, :-1] - 1
print('x \n', x)
print()
print('x shape : ', x.shape)
print()

# expectation
# reshape((-1, 1) -> makes the mean a 3 x 1 column, so it can be subtracted from every days column later.
e_x = average(x, axis=1).reshape((-1, 1))
print('e_x \n', e_x.round(5))
print()

# covariance
# cov(x) -> divides T-1 (not T)
cv_x = cov(x)
print('cv_x \n', cv_x.round(5))

x 
 [[ 0.00061208 -0.00652895 -0.00225972 -0.00452968  0.00641216  0.0069852
  -0.00775441  0.01131257 -0.00406802  0.00183829]
 [ 0.00549296  0.02134871 -0.00340331 -0.00634058  0.00687344  0.01608823
   0.00335922  0.00813008 -0.00236935  0.00570383]
 [ 0.0061376  -0.00677874  0.006825   -0.00067857 -0.0020371   0.02585596
  -0.01193887 -0.00805542  0.00135347 -0.00810986]]

x shape :  (3, 10)

e_x 
 [[0.0002 ]
 [0.00549]
 [0.00026]]

cv_x 
 [[4.0e-05 1.0e-05 2.0e-05]
 [1.0e-05 7.0e-05 1.0e-05]
 [2.0e-05 1.0e-05 1.2e-04]]


2. Principal-component model in a 3-stock market:

Eigenvalues and Eigenvectors

In [14]:
# eigenvectors and eigenvalues
# eigh always returns eigenvalues in ascending order
lam, e = eigh(cv_x)

# reversing-resorting eigenvalues
lam = lam[::-1]
print('lam = ', lam.round(5))
print()

# sorted eigenvectors
e = e[:, ::-1]
# flip each eigenvector so its entries sum positive
e = e* ((e.sum(axis=0) >= 0) * 2 - 1)
print('e \n', e.round(2))

lam =  [1.3e-04 7.0e-05 3.0e-05]

e 
 [[ 0.26  0.22  0.94]
 [ 0.26  0.92 -0.29]
 [ 0.93 -0.32 -0.18]]


3. Principal-component model in a 3-stock market: optimal

Loadings and construction matrix

In [15]:
# optimal loadings
beta = e[:, :1]
print('beta \n', beta.round(2))
print()

# Factor construction matrix
gamma = e[:, :1].T
print('gamma = ', gamma.round(2))

beta 
 [[0.26]
 [0.26]
 [0.93]]

gamma =  [[0.26 0.26 0.93]]


4. Principal-component model in a 3-stock market: first principal component

The first principal factor

In [16]:
# expectation of first factor
# This is always 0 and just shows that taking expectations passes through a linear map
e_z_pc = gamma@(e_x - e_x)
print('e_z_pc = ', e_z_pc)
print()

# variance of first factor
# This should print the same value as 'lam[0]'.
# Thats the 'factor variance equals eigenvalue' facted checked from before.
var_z_pc = gamma@cv_x@gamma.T
print('var_z_pc = ', var_z_pc)
print()

# time series of target variables
# fixed so that its now stocks x days
# each line will print 1 stock's 10 day series
print('x_1_t = ', x[0, :].round(4))
print()
print('x_2_t = ', x[1, :].round(4))
print()
print('x_3_t = ', x[2, :].round(4))
print()

# realizations of the first factor
# gamma multiplies every centered day at once, giving the 10 day factor series.
z_pc = gamma@(x - e_x)
print('z_pc = ', z_pc.round(4))

e_z_pc =  [[0.]]

var_z_pc =  [[0.00012863]]

x_1_t =  [ 0.0006 -0.0065 -0.0023 -0.0045  0.0064  0.007  -0.0078  0.0113 -0.0041
  0.0018]

x_2_t =  [ 0.0055  0.0213 -0.0034 -0.0063  0.0069  0.0161  0.0034  0.0081 -0.0024
  0.0057]

x_3_t =  [ 0.0061 -0.0068  0.0068 -0.0007 -0.002   0.0259 -0.0119 -0.0081  0.0014
 -0.0081]

z_pc =  [[ 0.0056 -0.0042  0.0032 -0.0052 -0.0002  0.0283 -0.014  -0.0042 -0.0021
  -0.0073]]


5. Principal-component model ina 3-stock market:

Prediction

In [18]:
# projection matrix
beta_betaproj = beta@gamma

# expectation of principal-component prediction
# the prediction has the same mean as x
e_x_pc = e_x

# covariance of principal-component prediction
# the covariance of the projected data, PCv{X}P'
cv_x_pc = beta_betaproj@beta_betaproj.T
print('cv_x_pc \n', cv_x_pc.round(5))

cv_x_pc 
 [[0.06719 0.06738 0.24111]
 [0.06738 0.06756 0.24178]
 [0.24111 0.24178 0.86525]]


In [19]:
# Not in the lab notebook - To settle the rank question from the page
print('rank = ', matrix_rank(cv_x_pc))

rank =  1


6. Principal-component model in a 3-stock market:

R-Squared

This is the first eigenvalue over the total.
The page gets 55%.
This is the same as the homeworks formula - cumsum(lam)/sum(lam)

In [21]:
# r squared
r2 = lam[0]/sum(lam)

print('r2 = ', r2.round(4))

r2 =  0.5507


7. Principal-component model in a 3-stock market:

Residuals

In [22]:
# scaling/rotation transformation
# this stacks 2 linear maps into one 4x3 matrix
# the top 3 rows, I -betaY, turning X - E{X} into the residuals
# the bottom row, y, turns X-E{X} into the factor Z
b = r_[eye(3) - beta@gamma, gamma]

# covariance of residual and factor
# b@cv_x@b.T -> is the full 4x4 covariance vs residuals and factor together
# check the last row and column are zeros
# the top left 3x3 block has non-zero off diagonals
cv_epsz_pc = b@cv_x@b.T
print('cv_epsz_pc = ', cv_epsz_pc.round(5))
print()

# covariance of residuals
cr_eps1_eps2 = cv_epsz_pc[0,1]/(sqrt(cv_epsz_pc[0,0]*sqrt(cv_epsz_pc[1,1])))
print('cr_eps1_eps2 = ', cr_eps1_eps2.round(2))
print()

cr_eps1_eps3 = cv_epsz_pc[0,2]/(sqrt(cv_epsz_pc[0,0]*sqrt(cv_epsz_pc[2,2])))
print('cr_eps1_eps3 = ', cr_eps1_eps3.round(2))
print()

cr_eps2_eps3 = cv_epsz_pc[1,2]/(sqrt(cv_epsz_pc[1,1]*sqrt(cv_epsz_pc[2,2])))
print('cr_eps1_eps2 = ', cr_eps1_eps2.round(2))

cv_epsz_pc =  [[ 3.0e-05  1.0e-05 -1.0e-05 -0.0e+00]
 [ 1.0e-05  6.0e-05 -2.0e-05 -0.0e+00]
 [-1.0e-05 -2.0e-05  1.0e-05 -0.0e+00]
 [ 0.0e+00 -0.0e+00 -0.0e+00  1.3e-04]]

cr_eps1_eps2 =  0.01

cr_eps1_eps3 =  -0.03

cr_eps1_eps2 =  0.01
